In [0]:
%pip install databricks-labs-dqx

In [0]:
dbutils.library.restartPython()

In [0]:
# Verify install
import databricks.labs.dqx
print(f"DQX version: {databricks.labs.dqx.__version__}")

In [0]:
# Setup widgets
 
dbutils.widgets.text("catalog_name",    "food_inspection")
dbutils.widgets.text("bronze_schema",   "bronze")
dbutils.widgets.text("metadata_schema", "metadata")
 
catalog_name    = dbutils.widgets.get("catalog_name")
bronze_schema   = dbutils.widgets.get("bronze_schema")
metadata_schema = dbutils.widgets.get("metadata_schema")
 
spark.sql(f"USE CATALOG {catalog_name}")
 
print(f"Catalog: {catalog_name}")
print(f"Bronze schema: {bronze_schema}")

In [0]:
# Create profile results table
 
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.{metadata_schema}.dqx_profile_results (
        table_name          STRING,
        city                STRING,
        column_name         STRING,
        profiling_dimension STRING,
        metric_name         STRING,
        metric_value        STRING,
        profile_time        TIMESTAMP,
        created_date        DATE
    )
""")
 
spark.sql(f"DELETE FROM {catalog_name}.{metadata_schema}.dqx_profile_results WHERE created_date = current_date()")
print("dqx_profile_results ready (cleared today's stale entries)")

In [0]:
# Imports and helper function
 
from pyspark.sql.functions import (
    col, count, when, isnull, lit, length, trim,
    countDistinct, min as spark_min, max as spark_max,
    avg as spark_avg, stddev as spark_stddev,
    sum as spark_sum, current_timestamp, current_date,
    lower, size, split as spark_split, round as spark_round
)
 
def log_profile(table_name, city, column_name, dimension, metric_name, metric_value):
    safe_val = str(metric_value).replace("'", "''")
    spark.sql(f"""
        INSERT INTO {catalog_name}.{metadata_schema}.dqx_profile_results VALUES (
            '{table_name}', '{city}', '{column_name}', '{dimension}',
            '{metric_name}', '{safe_val}', current_timestamp(), current_date()
        )
    """)

In [0]:
# DQX Profile — Chicago
from databricks.labs.dqx.profiler.profiler import DQProfiler
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()
profiler = DQProfiler(ws)

chi_df = spark.table(f"{catalog_name}.{bronze_schema}.bronze_chicago")
chi_summary, chi_profiles = profiler.profile(chi_df)

print("=== DQX Chicago Summary ===")
display(chi_summary)

In [0]:
# DQX auto-generated rules — Chicago
from databricks.labs.dqx.profiler.generator import DQGenerator

generator = DQGenerator(ws)
chi_rules = generator.generate_dq_rules(chi_profiles)

print("=== Suggested DQ Rules for Chicago ===")
for rule in chi_rules:
    print(f"  {rule}")

In [0]:
# DQX Profile — Dallas
dal_df = spark.table(f"{catalog_name}.{bronze_schema}.bronze_dallas")
dal_summary, dal_profiles = profiler.profile(dal_df)

print("=== DQX Dallas Summary ===")
display(dal_summary)

In [0]:
# DQX auto-generated rules — Dallas
dal_rules = generator.generate_dq_rules(dal_profiles)

print("=== Suggested DQ Rules for Dallas ===")
for rule in dal_rules:
    print(f"  {rule}")

In [0]:
# MAGIC %md
# MAGIC ---
# MAGIC # PART A — CHICAGO PROFILING
 
# COMMAND ----------
 
# Load Chicago Bronze
 
chicago = spark.table(f"{catalog_name}.{bronze_schema}.bronze_chicago")
chi_total = chicago.count()
chi_cols = [c for c in chicago.columns if not c.startswith("_")]
 
print(f"Chicago Bronze — rows: {chi_total}, columns: {len(chicago.columns)}")
print(f"Data columns (excl. audit): {len(chi_cols)}")
log_profile("bronze_chicago", "Chicago", "_table_", "overview", "row_count", chi_total)
log_profile("bronze_chicago", "Chicago", "_table_", "overview", "column_count", len(chi_cols))

In [0]:
# Print Chicago schema
chicago.printSchema()

In [0]:
# Chicago: null + empty count per column
 
print("=== CHICAGO: Completeness ===\n")
 
chi_completeness = []
for c in chi_cols:
    null_count = chicago.filter(col(c).isNull() | (trim(col(c).cast("string")) == "")).count()
    null_pct = round(100 * null_count / chi_total, 2)
    fill_rate = round(100 - null_pct, 2)
 
    chi_completeness.append((c, null_count, null_pct, fill_rate))
    log_profile("bronze_chicago", "Chicago", c, "completeness", "null_count", null_count)
    log_profile("bronze_chicago", "Chicago", c, "completeness", "null_pct", null_pct)
    log_profile("bronze_chicago", "Chicago", c, "completeness", "fill_rate", fill_rate)
 
chi_completeness_df = spark.createDataFrame(
    chi_completeness, ["column_name", "null_count", "null_pct", "fill_rate"]
).orderBy(col("null_pct").desc())
 
display(chi_completeness_df)
 

In [0]:
# Chicago: Is Inspection_ID truly unique?
 
chi_inspection_ids = chicago.select("Inspection_ID").distinct().count()
chi_duplicate_ids = chi_total - chi_inspection_ids
 
print(f"Total rows: {chi_total}")
print(f"Distinct Inspection_ID: {chi_inspection_ids}")
print(f"Duplicate Inspection_IDs: {chi_duplicate_ids}")
 
log_profile("bronze_chicago", "Chicago", "Inspection_ID", "uniqueness", "distinct_count", chi_inspection_ids)
log_profile("bronze_chicago", "Chicago", "Inspection_ID", "uniqueness", "duplicate_count", chi_duplicate_ids)

In [0]:
# Chicago: License number = 0 (suspicious)
chi_zero_license = chicago.filter(col("License_num") == 0).count()
print(f"License_num = 0: {chi_zero_license}")
log_profile("bronze_chicago", "Chicago", "License_num", "validity", "zero_license", chi_zero_license)

In [0]:
# Chicago: Cardinality of categorical columns
 
print("=== CHICAGO: Cardinality ===\n")
 
for c in ["Facility_Type", "Risk", "Inspection_Type", "Results", "City", "State"]:
    if c in chicago.columns:
        d = chicago.select(countDistinct(col(c))).collect()[0][0]
        print(f"  {c}: {d} distinct values")
        log_profile("bronze_chicago", "Chicago", c, "uniqueness", "cardinality", d)

In [0]:
# MAGIC %md
# MAGIC ### A3. Validity — Format & Range Checks
 
# COMMAND ----------
 
# Chicago: Zip code format — must be 5-digit or ZIP+4
 
chi_valid_zip = chicago.filter(
    col("Zip").isNotNull() &
    col("Zip").cast("string").rlike("^\\d{5}(-\\d{4})?$")
).count()
chi_null_zip = chicago.filter(col("Zip").isNull()).count()
chi_invalid_zip = chi_total - chi_valid_zip - chi_null_zip
 
print(f"Zip — valid: {chi_valid_zip}, null: {chi_null_zip}, invalid format: {chi_invalid_zip}")
 
log_profile("bronze_chicago", "Chicago", "Zip", "validity", "valid_format", chi_valid_zip)
log_profile("bronze_chicago", "Chicago", "Zip", "validity", "null_count", chi_null_zip)
log_profile("bronze_chicago", "Chicago", "Zip", "validity", "invalid_format", chi_invalid_zip)
 

In [0]:
print("=== CHICAGO: Zip length distribution ===")
display(chicago.filter(col("Zip").isNotNull())
    .withColumn("zip_length", length(col("Zip").cast("string")))
    .groupBy("zip_length").count().orderBy("zip_length"))

In [0]:
# Chicago: Inspection Date null check
 
chi_date_nonnull = chicago.filter(col("Inspection_Date").isNotNull()).count()
chi_date_null = chi_total - chi_date_nonnull
 
print(f"Inspection_Date — non-null: {chi_date_nonnull}, null: {chi_date_null}")
log_profile("bronze_chicago", "Chicago", "Inspection_Date", "validity", "non_null", chi_date_nonnull)
log_profile("bronze_chicago", "Chicago", "Inspection_Date", "validity", "null_count", chi_date_null)

In [0]:
# Chicago: Latitude/Longitude range validation
# Chicago coords should be approx 41.6-42.0 lat, -87.5 to -87.9 lon
 
chi_valid_coords = chicago.filter(
    col("Latitude").isNotNull() &
    (col("Latitude").cast("double").between(41.0, 43.0)) &
    (col("Longitude").cast("double").between(-89.0, -86.0))
).count()
chi_null_coords = chicago.filter(col("Latitude").isNull()).count()
chi_invalid_coords = chi_total - chi_valid_coords - chi_null_coords
 
print(f"Coords — valid range: {chi_valid_coords}, null: {chi_null_coords}, out of range: {chi_invalid_coords}")
 
log_profile("bronze_chicago", "Chicago", "Latitude", "validity", "valid_range", chi_valid_coords)
log_profile("bronze_chicago", "Chicago", "Latitude", "validity", "null_count", chi_null_coords)
log_profile("bronze_chicago", "Chicago", "Latitude", "validity", "out_of_range", chi_invalid_coords)

In [0]:
# Chicago: Value distributions for all key categorical columns

cat_cols = ["Results", "Risk", "Facility_Type", "Inspection_Type", "City", "State"]

for c in cat_cols:
    if c in chicago.columns:
        print(f"\n=== {c} ({chicago.select(c).distinct().count()} distinct) ===")
        display(chicago.groupBy(c).count()
            .withColumn("pct", spark_round(100.0 * col("count") / chi_total, 2))
            .orderBy(col("count").desc()).limit(520))

In [0]:
# Chicago: Out-of-state records
print("=== CHICAGO: Records outside Illinois ===")
display(chicago.filter(col("State") != "IL")
    .groupBy("City", "State", "Zip").count()
    .orderBy(col("count").desc()))

chi_non_il = chicago.filter(col("State") != "IL").count()
print(f"Non-IL records: {chi_non_il} out of {chi_total}")
log_profile("bronze_chicago", "Chicago", "State", "consistency", "non_illinois_records", chi_non_il)

In [0]:
# ASSIGNMENT RULE: PASS results should NOT have Urgent/Critical violations
 
chi_pass_urgent = chicago.filter(
    (col("Results") == "Pass") &
    (lower(col("Violations")).rlike("urgent|critical"))
).count()
 
print(f"PASS with Urgent/Critical violations: {chi_pass_urgent}")
print(f"  → These will be DROPPED in Silver (invalid combination per assignment)")
log_profile("bronze_chicago", "Chicago", "Results", "consistency", "pass_with_urgent_critical", chi_pass_urgent)

In [0]:
# Chicago: Inspections with NULL or empty violations blob
 
chi_no_violations = chicago.filter(
    col("Violations").isNull() | (trim(col("Violations")) == "")
).count()
 
print(f"Inspections with no violations text: {chi_no_violations}")
print(f"  → These will get a default 'No Violation' row in Silver")
log_profile("bronze_chicago", "Chicago", "Violations", "consistency", "missing_violations", chi_no_violations)

In [0]:
# Chicago: Sample violations blob — verify pipe-delimited + code.description format
 
print("=== CHICAGO: Sample violations (verify structure) ===")
display(chicago.select("Inspection_ID", "Violations")
    .filter(col("Violations").isNotNull())
    .limit(5))


# Chicago: Violation count per inspection (based on pipe delimiter)
 
chi_viol_counts = (chicago
    .filter(col("Violations").isNotNull())
    .withColumn("n_violations", size(spark_split(col("Violations"), "\\|")))
)
 
print("=== CHICAGO: Violations per inspection stats ===")
display(chi_viol_counts.select(
    spark_min("n_violations").alias("min_violations"),
    spark_max("n_violations").alias("max_violations"),
    spark_round(spark_avg("n_violations"), 1).alias("avg_violations")
))
 

In [0]:
# Chicago: Violations per inspection distribution
 
print("=== CHICAGO: Violations per inspection distribution ===")
display(chi_viol_counts.groupBy("n_violations").count().orderBy("n_violations"))

In [0]:
# Chicago: Date range
 
print("=== CHICAGO: Date range ===")
display(chicago.filter(col("Inspection_Date").isNotNull()).select(
    spark_min("Inspection_Date").alias("earliest_date"),
    spark_max("Inspection_Date").alias("latest_date"),
    countDistinct("Inspection_Date").alias("distinct_dates")
))

In [0]:
# Load Dallas Bronze
 
dallas = spark.table(f"{catalog_name}.{bronze_schema}.bronze_dallas")
dal_total = dallas.count()
 
dal_core_cols = [c for c in dallas.columns
                 if not c.startswith("Violation") and not c.startswith("_")]
dal_violation_cols = [c for c in dallas.columns if c.startswith("Violation")]
 
print(f"Dallas Bronze — rows: {dal_total}, total columns: {len(dallas.columns)}")
print(f"Core columns: {len(dal_core_cols)}")
print(f"Violation columns: {len(dal_violation_cols)}")
 
log_profile("bronze_dallas", "Dallas", "_table_", "overview", "row_count", dal_total)
log_profile("bronze_dallas", "Dallas", "_table_", "overview", "total_columns", len(dallas.columns))
log_profile("bronze_dallas", "Dallas", "_table_", "overview", "violation_columns", len(dal_violation_cols))

In [0]:
# Print Dallas schema (core columns only to keep it readable)
print("=== DALLAS: Core columns schema ===")
for c in dal_core_cols:
    dtype = str(dallas.schema[c].dataType)
    print(f"  {c}: {dtype}")

In [0]:
# Dallas: null + empty count per core column
 
print("=== DALLAS: Completeness (core columns) ===\n")
 
dal_completeness = []
for c in dal_core_cols:
    null_count = dallas.filter(col(f"`{c}`").isNull() | (trim(col(f"`{c}`").cast("string")) == "")).count()
    null_pct = round(100 * null_count / dal_total, 2)
    fill_rate = round(100 - null_pct, 2)
 
    dal_completeness.append((c, null_count, null_pct, fill_rate))
    log_profile("bronze_dallas", "Dallas", c, "completeness", "null_count", null_count)
    log_profile("bronze_dallas", "Dallas", c, "completeness", "null_pct", null_pct)
    log_profile("bronze_dallas", "Dallas", c, "completeness", "fill_rate", fill_rate)
 
dal_completeness_df = spark.createDataFrame(
    dal_completeness, ["column_name", "null_count", "null_pct", "fill_rate"]
).orderBy(col("null_pct").desc())
 
display(dal_completeness_df)

In [0]:
# Dallas: Fully duplicate rows
 
dal_deduped = dallas.dropDuplicates(dal_core_cols).count()
dal_full_dupes = dal_total - dal_deduped
 
print(f"Fully duplicate rows: {dal_full_dupes}")
log_profile("bronze_dallas", "Dallas", "_table_", "uniqueness", "full_duplicate_rows", dal_full_dupes)

In [0]:
# Dallas has NO native unique ID
# Check if name + date + address combos are unique
 
dal_natural_key = dallas.select(
    "Restaurant_Name", "Inspection_Date", "Zip_Code"
).distinct().count()
dal_natural_dupes = dal_total - dal_natural_key
 
print(f"Total rows: {dal_total}")
print(f"Distinct (name + date + address): {dal_natural_key}")
print(f"Duplicate combos: {dal_natural_dupes}")
print(f"  → MD5 hash will be generated in Silver from name + date + address + type")
 
log_profile("bronze_dallas", "Dallas", "_composite_key_", "uniqueness", "distinct_natural_keys", dal_natural_key)
log_profile("bronze_dallas", "Dallas", "_composite_key_", "uniqueness", "duplicate_natural_keys", dal_natural_dupes)

In [0]:
# Dallas has NO native unique ID
# Check which key combination gives us uniqueness

# 3-column key
key_3 = dallas.select(
    "Restaurant_Name", "Inspection_Date", "Street_Address"
).distinct().count()

# 4-column key (adding inspection_type)
key_4 = dallas.select(
    "Restaurant_Name", "Inspection_Date", "Street_Address", "Inspection_Type"
).distinct().count()

# 4-column key after dropping full duplicates
deduped = dallas.dropDuplicates([c for c in dallas.columns if not c.startswith("_")])
key_4_deduped = deduped.select(
    "Restaurant_Name", "Inspection_Date", "Street_Address", "Inspection_Type"
).distinct().count()

print(f"Total rows: {dal_total}")
print(f"Distinct (name + date + address):          {key_3}  → {dal_total - key_3} dupes")
print(f"Distinct (name + date + address + type):    {key_4}  → {dal_total - key_4} dupes")
print(f"After dedup + 4-col key:                    {key_4_deduped}  → {deduped.count() - key_4_deduped} dupes")
print(f"  → MD5 hash will use name + date + address + type in Silver")

log_profile("bronze_dallas", "Dallas", "_composite_key_", "uniqueness", "dupes_3col", dal_total - key_3)
log_profile("bronze_dallas", "Dallas", "_composite_key_", "uniqueness", "dupes_4col", dal_total - key_4)
log_profile("bronze_dallas", "Dallas", "_composite_key_", "uniqueness", "dupes_4col_after_dedup", deduped.count() - key_4_deduped)

In [0]:
# Test adding inspection_score to the key
key_5 = dallas.select(
    "Restaurant_Name", "Inspection_Date", "Street_Address", "Inspection_Type", "Inspection_Score"
).distinct().count()

# After dedup + 5-col key
key_5_deduped = deduped.select(
    "Restaurant_Name", "Inspection_Date", "Street_Address", "Inspection_Type", "Inspection_Score"
).distinct().count()

print(f"Distinct (name+date+address+type+score):       {key_5}  → {dal_total - key_5} dupes")
print(f"After dedup + 5-col key:                        {key_5_deduped}  → {deduped.count() - key_5_deduped} dupes")

In [0]:
# Dallas: Inspection Score — descriptive stats
 
dal_score_stats = dallas.filter(col("Inspection_Score").isNotNull()).select(
    spark_min(col("Inspection_Score").cast("int")).alias("min_score"),
    spark_max(col("Inspection_Score").cast("int")).alias("max_score"),
    spark_round(spark_avg(col("Inspection_Score").cast("int")), 2).alias("avg_score"),
    spark_round(spark_stddev(col("Inspection_Score").cast("int")), 2).alias("stddev_score")
).collect()[0]
 
print(f"Inspection Score stats:")
print(f"  min: {dal_score_stats['min_score']}")
print(f"  max: {dal_score_stats['max_score']}")
print(f"  avg: {dal_score_stats['avg_score']}")
print(f"  stddev: {dal_score_stats['stddev_score']}")
 
log_profile("bronze_dallas", "Dallas", "Inspection_Score", "validity", "min", dal_score_stats['min_score'])
log_profile("bronze_dallas", "Dallas", "Inspection_Score", "validity", "max", dal_score_stats['max_score'])
log_profile("bronze_dallas", "Dallas", "Inspection_Score", "validity", "avg", dal_score_stats['avg_score'])
log_profile("bronze_dallas", "Dallas", "Inspection_Score", "validity", "stddev", dal_score_stats['stddev_score'])

In [0]:
# Dallas: Score distribution buckets
 
print("=== DALLAS: Score distribution ===")
display(spark.sql(f"""
    SELECT
        CASE
            WHEN CAST(Inspection_Score AS INT) < 0    THEN 'Negative'
            WHEN CAST(Inspection_Score AS INT) < 60   THEN '0-59'
            WHEN CAST(Inspection_Score AS INT) < 70   THEN '60-69'
            WHEN CAST(Inspection_Score AS INT) < 80   THEN '70-79'
            WHEN CAST(Inspection_Score AS INT) < 90   THEN '80-89'
            WHEN CAST(Inspection_Score AS INT) <= 100  THEN '90-100'
            ELSE 'NULL'
        END AS score_bucket,
        COUNT(*) AS cnt,
        ROUND(100.0 * COUNT(*) / {dal_total}, 2) AS pct
    FROM {catalog_name}.{bronze_schema}.bronze_dallas
    GROUP BY 1 ORDER BY 1
"""))

In [0]:
# Dallas: Zip code format validation
 
dal_valid_zip = dallas.filter(
    col("Zip_Code").isNotNull() &
    col("Zip_Code").cast("string").rlike("^\\d{5}(-\\d{4})?$")
).count()
dal_null_zip = dallas.filter(col("Zip_Code").isNull()).count()
dal_invalid_zip = dal_total - dal_valid_zip - dal_null_zip
 
print(f"Zip Code — valid: {dal_valid_zip}, null: {dal_null_zip}, invalid: {dal_invalid_zip}")
 
log_profile("bronze_dallas", "Dallas", "Zip_Code", "validity", "valid_format", dal_valid_zip)
log_profile("bronze_dallas", "Dallas", "Zip_Code", "validity", "null_count", dal_null_zip)
log_profile("bronze_dallas", "Dallas", "Zip_Code", "validity", "invalid_format", dal_invalid_zip)

In [0]:
# Dallas: Inspection Date null check
 
dal_date_nonnull = dallas.filter(col("Inspection_Date").isNotNull()).count()
dal_date_null = dal_total - dal_date_nonnull
 
print(f"Inspection_Date — non-null: {dal_date_nonnull}, null: {dal_date_null}")
log_profile("bronze_dallas", "Dallas", "Inspection_Date", "validity", "non_null", dal_date_nonnull)
log_profile("bronze_dallas", "Dallas", "Inspection_Date", "validity", "null_count", dal_date_null)

In [0]:
# Dallas: Cardinality for key columns

dal_cat_cols = ["Restaurant_Name", "Inspection_Type", "Zip_Code", 
                "Street_Direction", "Street_Type", "Inspection_Month", 
                "Inspection_Year"]

print("=== DALLAS: Cardinality ===\n")
for c in dal_cat_cols:
    if c in dallas.columns:
        d = dallas.select(countDistinct(col(c))).collect()[0][0]
        print(f"  {c}: {d} distinct values")
        log_profile("bronze_dallas", "Dallas", c, "uniqueness", "cardinality", d)

In [0]:
# Dallas: Value distributions for all key categorical columns

dal_cat_cols = ["Inspection_Type", "Zip_Code", "city", "State"]

for c in dal_cat_cols:
    if c in dallas.columns:
        print(f"\n=== {c} ({dallas.select(c).distinct().count()} distinct) ===")
        display(dallas.groupBy(c).count()
            .withColumn("pct", spark_round(100.0 * col("count") / dal_total, 2))
            .orderBy(col("count").desc()).limit(15))

In [0]:
# Dallas: Lat Long Location parseability
 
dal_parseable_coords = dallas.filter(
    col("Lat_Long_Location").rlike("^\\([-0-9.]+,\\s*[-0-9.]+\\)$")
).count()
dal_null_coords = dallas.filter(col("Lat_Long_Location").isNull()).count()
dal_unparseable = dal_total - dal_parseable_coords - dal_null_coords
 
print(f"Lat_Long_Location — parseable: {dal_parseable_coords}, null: {dal_null_coords}, unparseable: {dal_unparseable}")
 
log_profile("bronze_dallas", "Dallas", "Lat_Long_Location", "validity", "parseable", dal_parseable_coords)
log_profile("bronze_dallas", "Dallas", "Lat_Long_Location", "validity", "null_count", dal_null_coords)
log_profile("bronze_dallas", "Dallas", "Lat_Long_Location", "validity", "unparseable", dal_unparseable)

In [0]:
# Dallas: Violation density — how many of 25 slots used per row
 
violation_desc_cols = [c for c in dallas.columns if "Violation_Description" in c]
print(f"Found {len(violation_desc_cols)} violation description columns")
 
density_expr = sum(
    when(col(f"`{c}`").isNotNull() & (trim(col(f"`{c}`").cast("string")) != ""), 1).otherwise(0)
    for c in violation_desc_cols
)
 
dallas_density = dallas.withColumn("n_violations", density_expr)
 
# COMMAND ----------
 
# Dallas: Violations per inspection distribution
 
print("=== DALLAS: Violations per inspection distribution ===")
display(dallas_density.groupBy("n_violations").count().orderBy("n_violations"))

In [0]:
# ASSIGNMENT RULE: Score >= 90 should have <= 3 violations
 
bad_score_violation = dallas_density.filter(
    (col("Inspection_Score").cast("int") >= 90) & (col("n_violations") > 3)
).count()
 
print(f"Score >= 90 with > 3 violations (VR-008): {bad_score_violation}")
print(f"  → These ENTIRE inspections will be DROPPED in Silver")
log_profile("bronze_dallas", "Dallas", "Inspection_Score", "consistency", "vr008_violations", bad_score_violation)

In [0]:
# Dallas: Check for Urgent/Critical terms in any violation description
 
urgent_critical_count = 0
for c in violation_desc_cols:
    uc = dallas.filter(lower(col(f"`{c}`")).rlike("urgent|critical")).count()
    urgent_critical_count += uc
 
print(f"Total violation entries containing Urgent/Critical: {urgent_critical_count}")
log_profile("bronze_dallas", "Dallas", "Violations", "consistency", "urgent_critical_count", urgent_critical_count)
 

In [0]:
# Dallas: Address component coverage
 
print("=== DALLAS: Address component coverage ===")
display(spark.sql(f"""
    SELECT
        SUM(CASE WHEN Street_Number    IS NOT NULL AND trim(Street_Number) != '' THEN 1 ELSE 0 END) AS has_street_number,
        SUM(CASE WHEN Street_Direction IS NOT NULL AND trim(Street_Direction) != '' THEN 1 ELSE 0 END) AS has_street_direction,
        SUM(CASE WHEN Street_Name      IS NOT NULL AND trim(Street_Name) != '' THEN 1 ELSE 0 END) AS has_street_name,
        SUM(CASE WHEN Street_Type      IS NOT NULL AND trim(Street_Type) != '' THEN 1 ELSE 0 END) AS has_street_type,
        SUM(CASE WHEN Street_Unit      IS NOT NULL AND trim(Street_Unit) != '' THEN 1 ELSE 0 END) AS has_street_unit,
        COUNT(*) AS total
    FROM {catalog_name}.{bronze_schema}.bronze_dallas
"""))
 
# COMMAND ----------
 
# Dallas: Column 20 double-space typo check
 
print("=== DALLAS: Column 20 memo typo check ===")
 
memo_variants = [c for c in dallas.columns if "Memo" in c and "20" in c]
print(f"Actual column 20 memo name(s): {memo_variants}")
print(f"  → Handle defensively in Silver unpivot loop")
log_profile("bronze_dallas", "Dallas", "Violation_Memo_20", "consistency", "actual_column_name", str(memo_variants))

In [0]:
# Dallas: Date range
 
print("=== DALLAS: Date range ===")
display(dallas.filter(col("Inspection_Date").isNotNull()).select(
    spark_min("Inspection_Date").alias("earliest_date"),
    spark_max("Inspection_Date").alias("latest_date"),
    countDistinct("Inspection_Date").alias("distinct_dates")
))

In [0]:
# Dallas: Check for non-DFW zip codes
display(dallas.filter(
    ~col("Zip_Code").rlike("^(750|751|752|760|761)") & 
    col("Zip_Code").isNotNull()
).select("Restaurant_Name", "Zip_Code", "Street_Address")
.groupBy("Zip_Code").count()
.orderBy(col("count").desc()))

dal_non_dfw_zips = dallas.filter(
    ~col("Zip_Code").rlike("^(750|751|752|760|761)") & 
    col("Zip_Code").isNotNull()
).count()
print(f"Non-DFW zip codes: {dal_non_dfw_zips}")

In [0]:
print("=" * 90)
print(f"{'Attribute':<28} {'Chicago':<30} {'Dallas':<30}")
print("=" * 90)
 
comparison = [
    ("Unique ID",           "Inspection_ID (native)",    "NONE - generate MD5 hash"),
    ("Restaurant Name",     "DBA_Name",                  "Restaurant_Name"),
    ("AKA Name",            "AKA_Name",                  "NOT AVAILABLE"),
    ("License",             "License_num",               "NOT AVAILABLE"),
    ("Facility Type",       "Facility_Type (granular)",  "NOT AVAILABLE"),
    ("Risk Category",       "Risk (Risk 1/2/3)",         "NOT AVAILABLE"),
    ("Address",             "Address (single field)",    "5 street components"),
    ("City",                "City column",               "NOT AVAILABLE - default DALLAS"),
    ("State",               "State column",              "NOT AVAILABLE - default TX"),
    ("Zip",                 "Zip (int/string)",          "Zip_Code (string)"),
    ("Inspection Date",     "Inspection_Date",           "Inspection_Date"),
    ("Inspection Type",     "Inspection_Type",           "Inspection_Type"),
    ("Inspection Result",   "Results",                   "NOT AVAILABLE"),
    ("Inspection Score",    "NOT AVAILABLE - derive",    "Inspection_Score (native)"),
    ("Violations",          "Pipe-delimited blob",       "25 column groups (wide)"),
    ("Violation Points",    "NOT AVAILABLE",             "Violation_Points_1..25"),
    ("Lat/Long",            "Separate columns",          "Combined (lat,long) string"),
    ("Month/Year",          "NOT AVAILABLE",             "Inspection_Month/_Year"),
]
 
for attr, chi, dal in comparison:
    print(f"  {attr:<28} {chi:<30} {dal:<30}")